# Praktikum 3: Exact NN dan HNSW pada data kafe

In [1]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
np.random.seed(42)
import pandas as pd
df = pd.read_csv('../../dataset_pbl/cafes.csv')
features = ['latitude', 'longitude', 'rating', 'review_count', 'price_min']
df[features] = df[features].fillna(df[features].mean())
dim = 2
data = df[features[:dim]].to_numpy(dtype=np.float32)
num_elements = len(data)

# Query point
query = data[:1].copy()
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.perf_counter()
distances, indices = nn.kneighbors(query)
end = time.perf_counter()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)  

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.set_num_threads(4)
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.perf_counter()
labels, distances = p.knn_query(query, k=k)
end = time.perf_counter()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

=== Exact NN ===
Indices: [[ 0 20 85 28 16]]
Distances: [[0.         0.00091186 0.00131403 0.00225466 0.00248853]]
Waktu: 1.9654513000004954 detik

=== HNSW ===
Indices: [[ 0 20 85 28 16]]
Distances: [[0.0000000e+00 8.3148279e-07 1.7266887e-06 5.0835133e-06 6.1927867e-06]]
Waktu: 0.0001609999999345746 detik


In [2]:
import pandas as pd
from IPython.display import display

def uji_hnsw(metric='euclidean', num_elements=1000, dim=2):
    np.random.seed(42)
    data = df[features[:dim]].iloc[:num_elements].to_numpy(dtype=np.float32)
    query = data[:1].copy()
    k = 5

    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric)
    nn.fit(data)
    start = time.perf_counter()
    distances_exact, indices = nn.kneighbors(query)
    waktu_exact = time.perf_counter() - start

    p = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=100, M=16, random_seed=42)
    p.set_num_threads(4)
    p.add_items(data)
    p.set_ef(50)

    start = time.perf_counter()
    labels, distances_hnsw = p.knn_query(query, k=k)
    waktu_hnsw = time.perf_counter() - start

    idx_exact = indices[0].tolist()
    idx_hnsw = labels[0].tolist()
    return {
        'Distance Metrics': metric.capitalize(), 'Jumlah data': num_elements, 'Dimensi': dim,
        'Hasil Index terdekat Exact vs HNSW': f'{idx_exact}, {idx_hnsw}',
        'Waktu Exact vs HNSW (detik)': f'{waktu_exact:.8f}, {waktu_hnsw:.8f}'
    }

hasil_hnsw = []
for metric in ['euclidean', 'cosine']:
    for jumlah in [256, len(df)]:
        for dim in [2, 5]:
            hasil_hnsw.append(uji_hnsw(metric, jumlah, dim))
pd.DataFrame(hasil_hnsw)

,Distance Metrics,Jumlah data,Dimensi,Hasil Index terdekat Exact vs HNSW,Waktu Exact vs HNSW (detik)
0,Euclidean,256,2,"[0, 20, 85, 28, 16], [0, 20, 85, 28, 16]","0.00277800, 0.00004520"
1,Euclidean,256,5,"[0, 85, 225, 201, 115], [0, 85, 225, 201, 115]","0.00291850, 0.00004650"
2,Euclidean,302,2,"[0, 20, 85, 28, 16], [0, 20, 85, 28, 16]","0.00337590, 0.00005120"
3,Euclidean,302,5,"[0, 85, 225, 201, 115], [0, 85, 225, 201, 115]","0.00299440, 0.00003920"
4,Cosine,256,2,"[1, 4, 6, 8, 9], [92, 125, 165, 177, 244]","0.00464330, 0.00004780"
5,Cosine,256,5,"[0, 85, 225, 115, 201], [0, 85, 225, 115, 201]","0.00168730, 0.00002930"
6,Cosine,302,2,"[301, 1, 300, 298, 4], [23, 185, 224, 235, 244]","0.00140120, 0.00004830"
7,Cosine,302,5,"[0, 85, 225, 115, 201], [0, 85, 225, 115, 201]","0.00135740, 0.00003490"
